## 0. Colab Setup

Run this section first if you are in Google Colab. If you are running locally in Jupyter, these cells are no-ops and can be skipped.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_ROOT = "/content/drive/MyDrive/food-preservation-methods-predictors"

    import os
    if not os.path.exists(f"{PROJECT_ROOT}/notebooks/_shared_formulas.py"):
        # Private repo -- needs a GitHub Personal Access Token (repo scope) to clone.
        # Generate one at https://github.com/settings/tokens and paste it when prompted.
        # getpass keeps it out of notebook cell output/history.
        from getpass import getpass
        _gh_token = getpass("GitHub Personal Access Token: ")
        _clone_url = f"https://{_gh_token}@github.com/SafaeHaj/food-preservation-methods-predictors.git"
        print(f"Repo not found at {PROJECT_ROOT} -- cloning...")
        !git clone "{_clone_url}" "{PROJECT_ROOT}"
        del _gh_token, _clone_url
    else:
        print(f"Repo already present at {PROJECT_ROOT} -- pulling latest...")
        !cd "{PROJECT_ROOT}" && git pull


In [ ]:
import torch
if IN_COLAB:
    if torch.cuda.is_available():
        print(f"GPU available: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU "
              "before training CTGAN/TVAE -- this notebook is compute-heavy.")


In [ ]:
if IN_COLAB:
    %pip install -q sdv==1.37.2 sdmetrics ctgan


In [ ]:
os.path.exists("/content/drive/MyDrive/food-preservation-methods-predictors/notebooks/_shared_formulas.py")


# Synthetic Data Generation

Evaluate CTGAN and TVAE as synthetic data generators for the combined shelf-life dataset.

Key design choices:
- Derived columns (`initial_inhibition_factor`, `post_inhibition_factor`, `post_threshold_proximity_index_2(tpi)`) are **excluded from GAN training** and recomputed deterministically afterward.
- `experiment_id` and `ingredient_id` are **re-assigned after sampling** based on grouping keys, preserving logical structure.
- `data_source` is hard-coded to `"synthetic"` for all generated rows.
- `N_EXPERIMENTS` controls the size of the output (configurable at the top of Section 1).

## Table of Contents

1. [Setup and Imports](#1-setup-and-imports)
2. [Column Definitions](#2-column-definitions)
3. [Pre-processing for GAN](#3-pre-processing-for-gan)
4. [SDV Metadata](#4-sdv-metadata)
   - [4.5. Shared Utilities](#45-shared-utilities)
5. [Model Factory](#5-model-factory)
   - [5.5. Hyperparameter Search](#55-hyperparameter-search)
6. [Training Loop (HPO-tuned)](#6-training-loop-hpo-tuned)
   - [6.1. Conditional Generation Helper](#61-conditional-generation-helper)
7. [Post-processing: Re-assign IDs & Recompute Derived Columns](#7-post-processing-re-assign-ids--recompute-derived-columns)
   - [7.1. Post-Generation Validation](#71-post-generation-validation)
   - [7.2. Final Output](#72-final-output)
8. [TSTR Evaluation](#8-tstr-evaluation)
9. [Results Summary](#9-results-summary)
10. [Statistical Fidelity Check](#10-statistical-fidelity-check)

## 1. Setup and Imports

In [15]:
import warnings
warnings.filterwarnings("ignore")

import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import ks_2samp
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer, CopulaGANSynthesizer

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Configurable ─────────────────────────────────────────────────────────────
N_EXPERIMENTS = 1000   # number of unique synthetic experiment groups to produce
EPOCHS        = 300   # GAN training epochs
# ─────────────────────────────────────────────────────────────────────────────

ROOT          = Path(PROJECT_ROOT) if IN_COLAB else Path("..").resolve()
DATA_DIR      = ROOT / "data"
SYNTHETIC_DIR = DATA_DIR / "synthetic"
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)

# compute_inhibition_features lives in _shared_formulas.py so the three notebooks that
# recompute these derived columns (data_processing, data_filling, baranyi_growth_modelling)
# stay in lock-step and can never drift. Same import pattern as baranyi_growth_modelling.
sys.path.append(str(ROOT / "notebooks"))
from _shared_formulas import compute_inhibition_features

df_raw = pd.read_csv(DATA_DIR / "processed" / "combined_shelf_life.csv")
print(f"Loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")
df_raw.head(3)

## 2. Column Definitions

In [16]:
# Columns whose values are deterministically derived from other columns.
# These are EXCLUDED from GAN training and recomputed after sampling.
DERIVED_COLS = [
    "normalized_concentration_%_of_meat",   # no formula available; left NaN
    "initial_inhibition_factor",
    "post_inhibition_factor",
    "post_threshold_proximity_index_2(tpi)"
]

ID_COLS = ["experiment_id", "ingredient_id"]

# Columns the GAN trains on (source columns only)
GAN_COLS = [
    c for c in df_raw.columns
    if c not in DERIVED_COLS + ID_COLS + ["data_source"]
]

# Categorical vs numerical split for SDV metadata and TSTR preprocessing
CAT_COLS = [c for c in GAN_COLS if df_raw[c].dtype == object]

# Threshold/target-derived columns are censored, not missing-at-random: a NaN here means
# the sample never crossed that threshold during the observation window.
# These get a censored-state encoding (-1 + mask) in Section 3
# instead of median imputation, which would fabricate a crossing day/count that was never
# observed.
THRESHOLD_COLS = ["post_threshold_(day)", "post_threshold_count"]

NUM_COLS = [c for c in GAN_COLS if c not in CAT_COLS]

# Grouping key used to re-assign experiment_id after generation.
# Rows that share these values belong to the same synthetic experiment.
EXPERIMENT_KEY = [
    "meat_type", "indicators", "storage_condition_(°c)",
    "physical_hurdle_tech", "application_method", "hurdle_coordination",
]

print(f"GAN training columns ({len(GAN_COLS)}): {GAN_COLS}")
print(f"\nCategorical ({len(CAT_COLS)}): {CAT_COLS}")
print(f"\nNumerical   ({len(NUM_COLS)}): {NUM_COLS}")
print(f"\nCensored/threshold ({len(THRESHOLD_COLS)}): {THRESHOLD_COLS}")

## 3. Pre-processing for GAN

Drop derived columns and IDs so the GAN never sees them. Then:

1. **Impute missing values** so CTGAN never trains on raw `NaN`:
   - Numerical (non-threshold): median impute + a binary `<col>_missing` indicator column.
   - Categorical: fill with a consistent `"__missing__"` category (no indicator needed —
     `"__missing__"` is itself an explicit level CTGAN can learn to reproduce).
   - Threshold columns (`post_threshold_(day)`, `post_threshold_count`): encoded as `-1` + a binary `<col>_censored` indicator to avoid uninformed data fabrication.
2. **Scale** every numerical value column (including the now-imputed threshold columns) with
   `MinMaxScaler`, fit once on the imputed training frame and reused later to inverse-transform
   generated samples back to the original scale (Section 7).
3. **Oversample** via `conditional_matrix_sample` to balance rare categorical configurations
   and post-threshold survivors before fitting.

In [17]:
from sklearn.preprocessing import MinMaxScaler

MISSING_CAT_TOKEN = "__missing__"
NUM_COLS_TO_IMPUTE = [c for c in NUM_COLS if c not in THRESHOLD_COLS]
MISSING_INDICATOR_COLS = [f"{c}_missing" for c in NUM_COLS_TO_IMPUTE]
CENSORED_INDICATOR_COLS = [f"{c}_censored" for c in THRESHOLD_COLS]
INDICATOR_COLS = MISSING_INDICATOR_COLS + CENSORED_INDICATOR_COLS


def impute_missing(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Impute NaNs so the GAN never trains on raw missing values.

      - Categorical      -> fill with MISSING_CAT_TOKEN (an explicit, learnable category).
      - Numerical         -> median impute + <col>_missing binary indicator.
      - Threshold/target  -> censored, not MAR: fill with -1 + <col>_censored indicator
                              instead of a fabricated median (NaN here means "never
                              crossed this threshold", not "unknown value").

    Returns the imputed frame plus a dict of fitted impute values (medians) so the same
    values can be reused at inference/inverse-transform time if needed.
    """
    df = df.copy()
    impute_values: dict = {}

    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(MISSING_CAT_TOKEN)

    for col in NUM_COLS_TO_IMPUTE:
        if col not in df.columns:
            continue
        median = df[col].median()
        impute_values[col] = median
        df[f"{col}_missing"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(median)

    for col in THRESHOLD_COLS:
        if col not in df.columns:
            continue
        df[f"{col}_censored"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(-1)

    return df, impute_values


def scale_numeric(df: pd.DataFrame, num_cols: list[str]) -> tuple[pd.DataFrame, MinMaxScaler]:
    """Fit-transform NUM_COLS (value columns only, not the 0/1 indicator columns)."""
    df = df.copy()
    scaler = MinMaxScaler()
    cols = [c for c in num_cols if c in df.columns]
    df[cols] = scaler.fit_transform(df[cols])
    return df, scaler


def conditional_matrix_sample(
    df: pd.DataFrame,
    cat_cols: list[str],
    sparse_threshold: float = 0.02,
    has_post: np.ndarray | None = None,
    target_n: int | None = None,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    has_post: precomputed boolean array marking rows that had real post-threshold data. 
    Passed explicitly rather than derived from `df.notna()` inside
    this function, because by the time `df` reaches here the threshold columns have
    already been imputed (-1 + *_censored indicator) — notna() would be True for every
    row and the post-threshold-survivor oversampling weight would silently become a no-op.
    """
    rng = np.random.default_rng(random_state)
    target_n = target_n or len(df)

    col_weights = np.ones(len(df))
    for col in cat_cols:
        freq = df[col].fillna(MISSING_CAT_TOKEN).map(
            df[col].fillna(MISSING_CAT_TOKEN).value_counts(normalize=True)
        ).values
        col_weights *= 1.0 / (np.log1p(freq * len(df)))

    if has_post is None:
        has_post = np.zeros(len(df), dtype=bool)
    col_weights = np.where(has_post, col_weights * 2.0, col_weights)

    sparse_masks = {}
    for col in cat_cols:
        freq_map = df[col].fillna(MISSING_CAT_TOKEN).value_counts(normalize=True)
        sparse_cats = freq_map[freq_map < sparse_threshold].index.tolist()
        sparse_masks[col] = df[col].fillna(MISSING_CAT_TOKEN).isin(sparse_cats)

    n_sparse_rows = pd.DataFrame(sparse_masks).any(axis=1).sum()
    print(f"Rows touching at least one sparse category (<{sparse_threshold:.0%}): {n_sparse_rows}/{len(df)}")
    print(f"Rows with post-threshold data: {has_post.sum()}/{len(df)}")

    weights = col_weights / col_weights.sum()
    sampled_idx = rng.choice(len(df), size=target_n, replace=True, p=weights)
    return df.iloc[sampled_idx].reset_index(drop=True)


# Strip derived cols, IDs, and data_source
df_gan_source = df_raw.drop(columns=DERIVED_COLS + ID_COLS + ["data_source"], errors="ignore")

POST_THRESHOLD_COLS = [c for c in THRESHOLD_COLS if c in df_gan_source.columns]

# Capture post-threshold survivorship from the RAW (pre-imputation) frame — this signal
# would be lost once impute_missing() fills those columns with -1.
HAS_POST = (
    df_gan_source[POST_THRESHOLD_COLS].notna().any(axis=1).values
    if POST_THRESHOLD_COLS
    else np.zeros(len(df_gan_source), dtype=bool)
)

# Impute (median + indicator / -1 + censored indicator / __missing__ category), then scale.
df_gan_imputed, IMPUTE_VALUES = impute_missing(df_gan_source)
df_gan_scaled, NUM_SCALER = scale_numeric(df_gan_imputed, NUM_COLS)

GAN_TRAIN_COLS = GAN_COLS + INDICATOR_COLS

df_gan = conditional_matrix_sample(
    df_gan_scaled,
    cat_cols=CAT_COLS,
    sparse_threshold=0.02,
    has_post=HAS_POST,
    target_n=len(df_gan_scaled),
    random_state=42,
)

print(f"\nGAN training frame: {df_gan.shape}")
print(f"Indicator columns added: {INDICATOR_COLS}")
print("Missing post_threshold_(day) (should be 0, now encoded via *_censored):",
      df_gan["post_threshold_(day)"].isna().sum(), "/", len(df_gan))
print("post_threshold_(day)_censored rate:", df_gan["post_threshold_(day)_censored"].mean().round(3))

## 4. SDV Metadata

`SingleTableMetadata` captures the role of every column (categorical, numerical)
and is shared by all synthesisers. `df_gan` now also contains the `*_missing` /
`*_censored` indicator columns added in Section 3 — these are declared `categorical`
(they're 0/1 flags, not continuous quantities) so CTGAN treats them as discrete modes
rather than fitting a mixture-of-Gaussians to a two-point distribution.
`CAT_COLS` and `NUM_COLS` are passed explicitly rather than relying solely on
auto-detection, so the categorical/numerical split used for sampling matches the
one used everywhere else in this notebook.

In [18]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_gan)

# Explicitly declare all three axes rather than trusting auto-detection alone
# this categorical/numerical split is used everywhere else in this notebook
# (CAT_COLS / NUM_COLS from Section 2, INDICATOR_COLS from Section 3), so CTGAN's
# column-type assumptions can't drift from the rest of the pipeline.
for col in CAT_COLS:
    if col in df_gan.columns:
        metadata.update_column(col, sdtype="categorical")
for col in NUM_COLS:
    if col in df_gan.columns:
        metadata.update_column(col, sdtype="numerical")
for col in INDICATOR_COLS:
    if col in df_gan.columns:
        metadata.update_column(col, sdtype="categorical")

metadata.validate()
print("Metadata validated")
print(list(metadata.to_dict()["columns"].keys()))

## 4.5. Shared Utilities

In [19]:
def inverse_transform_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Undo the Section-3 MinMaxScaler so GAN output is back on its original scale."""
    df = df.copy()
    cols = [c for c in NUM_COLS if c in df.columns]
    df[cols] = NUM_SCALER.inverse_transform(df[cols])
    return df

def assign_ids(df: pd.DataFrame, experiment_id_offset: int = 0) -> pd.DataFrame:
    """
    Assign sequential experiment_id and ingredient_id to a synthetic dataframe.

    experiment_id  — one integer per unique combination of EXPERIMENT_KEY columns.
    ingredient_id  — within each experiment, one integer per unique ingredient value.
    """
    df = df.copy()

    # Fill NaN in key columns with a sentinel so groupby works cleanly
    key_filled = df[EXPERIMENT_KEY].fillna("__missing__")
    exp_map = {
        combo: i + 1 + experiment_id_offset
        for i, combo in enumerate(key_filled.drop_duplicates().itertuples(index=False, name=None))
    }
    df["experiment_id"] = key_filled.apply(lambda row: exp_map[tuple(row)], axis=1)

    # ingredient_id within each experiment
    ingredient_ids = []
    for exp_id, group in df.groupby("experiment_id", sort=False):
        ing_map: dict = {}
        counter = 1
        ids = []
        for ing in group["ingredient"].fillna("__missing__"):
            if ing not in ing_map:
                ing_map[ing] = counter
                counter += 1
            ids.append(ing_map[ing])
        ingredient_ids.extend(ids)
    df["ingredient_id"] = ingredient_ids

    return df


def postprocess(df_syn: pd.DataFrame) -> pd.DataFrame:
    """Full post-processing pipeline for a raw GAN sample."""
    df = df_syn.copy()

    # Step 1: undo scaling — every downstream computation expects original units
    df = inverse_transform_numeric(df)

    # Step 2: drop training-only indicator columns (kept generated values, per design
    # goal of maximizing non-null synthetic data rather than re-introducing NaNs)
    df = df.drop(columns=[c for c in INDICATOR_COLS if c in df.columns])

    # Step 3: IDs — offset from real data max so they never collide
    exp_offset = int(df_raw["experiment_id"].max())
    df = assign_ids(df, experiment_id_offset=exp_offset)

    # Step 4: recompute derived columns
    df = compute_inhibition_features(df)
    df["normalized_concentration_%_of_meat"] = np.nan  # no formula available

    # label
    df["data_source"] = "synthetic"

    # Reorder to match real data schema exactly
    df = df.reindex(columns=df_raw.columns)
    return df

STRICT_NOVELTY = False  # if True, also reject combinations never seen in real data
PACKAGING_TUPLE_COLS = ["packaging_class", "packaging_material", "container_type"]
HURDLE_TUPLE_COLS = ["physical_hurdle_tech", "application_method", "hurdle_coordination"]

def rule_hybrid_without_hurdle(df: pd.DataFrame) -> pd.Series:
    """hurdle_coordination claims multiple hurdles but neither axis names one."""
    return (
        (df["hurdle_coordination"] == "hybrid_simultaneous")
        & df["physical_hurdle_tech"].isna()
        & df["application_method"].isna()
    )

HARD_REJECT_RULES = {
    "hybrid_coordination_without_hurdle": rule_hybrid_without_hurdle,
}


def discovered_whitelist(df_real: pd.DataFrame, tuple_cols: list[str]) -> set:
    """Observed combinations of tuple_cols in the real data — the data-driven novelty check."""
    cols = [c for c in tuple_cols if c in df_real.columns]
    return set(df_real[cols].fillna("__missing__").drop_duplicates().itertuples(index=False, name=None))


def flag_novel_combinations(df: pd.DataFrame, df_real: pd.DataFrame, tuple_cols: list[str]) -> pd.Series:
    cols = [c for c in tuple_cols if c in df.columns]
    whitelist = discovered_whitelist(df_real, tuple_cols)
    tuples = df[cols].fillna("__missing__").apply(tuple, axis=1)
    return ~tuples.isin(whitelist)


def clip_to_realistic_bounds(df: pd.DataFrame, df_real: pd.DataFrame, num_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in num_cols:
        if col not in df.columns or col not in df_real.columns:
            continue
        lo, hi = df_real[col].quantile([0.01, 0.99])
        df[col] = df[col].clip(lower=lo, upper=hi)
    return df


def validate_synthetic(df_syn: pd.DataFrame, df_real: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Full validation pipeline: gas reconstruction, hard rejection, novelty report, clipping."""
    n_generated = len(df_syn)
    df = df_syn.copy()
    rejection_masks = {}

    packaging_novel = flag_novel_combinations(df, df_real, PACKAGING_TUPLE_COLS)
    hurdle_novel = flag_novel_combinations(df, df_real, HURDLE_TUPLE_COLS)
    novel_combo = packaging_novel | hurdle_novel
    if STRICT_NOVELTY:
        rejection_masks["novel_combination"] = novel_combo

    reject_mask = pd.Series(False, index=df.index)
    for mask in rejection_masks.values():
        reject_mask |= mask

    df_retained = df.loc[~reject_mask].reset_index(drop=True)
    df_retained = clip_to_realistic_bounds(df_retained, df_real, NUM_COLS)

    report = {
        "n_generated": n_generated,
        "n_retained": len(df_retained),
        "n_rejected": int(reject_mask.sum()),
        "rejection_counts": {name: int(mask.sum()) for name, mask in rejection_masks.items()},
        "novel_combination_count": int(novel_combo.sum()),  # always reported, even if not rejected
    }
    return df_retained, report


def validation_report(report: dict) -> None:
    print(f"Generated: {report['n_generated']}")
    print(f"Retained:  {report['n_retained']} ({report['n_retained'] / report['n_generated']:.1%})")
    print(f"Rejected:  {report['n_rejected']} ({report['n_rejected'] / report['n_generated']:.1%})")
    print("\nRejection breakdown:")
    for rule_name, count in report["rejection_counts"].items():
        print(f"  {rule_name:<35} {count:>6}  ({count / report['n_generated']:.1%})")
    print(f"\nNovel (unseen-in-real) combinations observed: {report['novel_combination_count']} "
          f"({report['novel_combination_count'] / report['n_generated']:.1%})"
          f"{' — rejected (STRICT_NOVELTY=True)' if STRICT_NOVELTY else ' — reported only, not rejected'}")


# ── 8.x: TSTR evaluation helpers ──────────────
TARGET = "post_threshold_proximity_index_2(tpi)"

DROP_COLS    = ID_COLS + DERIVED_COLS + ["data_source"]
FEATURE_COLS = [c for c in df_raw.columns if c not in DROP_COLS]
CAT_FEATURES = [c for c in FEATURE_COLS if df_raw[c].dtype == object]
NUM_FEATURES = [c for c in FEATURE_COLS if c not in CAT_FEATURES]


def make_pipeline() -> Pipeline:
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_FEATURES),
        ("num", "passthrough", NUM_FEATURES),
    ])
    model = HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05, max_depth=6, random_state=42,
    )
    return Pipeline([("pre", pre), ("model", model)])


def evaluate(pipe: Pipeline, X_test: pd.DataFrame, y_test: pd.Series) -> dict:
    y_pred = pipe.predict(X_test)
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "MAE":  float(mean_absolute_error(y_test, y_pred)),
        "R2":   float(r2_score(y_test, y_pred)),
    }


def prepare_split(df: pd.DataFrame):
    d = df.dropna(subset=[TARGET]).copy()
    available_features = [c for c in FEATURE_COLS if c in d.columns]
    return d[available_features], d[TARGET]


# ── 10.x: Statistical fidelity helper  ────────
def ks_fidelity(df_real: pd.DataFrame, df_syn: pd.DataFrame) -> pd.Series:
    stats = {}
    for col in NUM_COLS:   # GAN numerical columns (source columns only)
        if col not in df_syn.columns:
            continue
        r = df_real[col].dropna()
        s = df_syn[col].dropna()
        if len(r) < 2 or len(s) < 2:
            continue
        stat, _ = ks_2samp(r, s)
        stats[col] = round(stat, 4)
    return pd.Series(stats, name="KS statistic")


print("Shared utilities ready: postprocess, validate_synthetic, make_pipeline/evaluate/prepare_split, ks_fidelity.")

## 5. Model Factory

All three synthesisers are built with shared training improvements:

- **WGAN-GP** — `pac=1` disables PAC-GAN grouping so the gradient penalty is applied per individual sample (CTGAN/CopulaGAN).
- **TTUR** (Two Time-Scale Update Rule) — discriminator lr is set to 4× the generator lr (8e-4 vs 2e-4) (CTGAN/CopulaGAN).
- **Spectral Normalization** — every `Linear` layer is spectrally normalized. CopulaGAN builds its own internal CTGAN instance, so it inherits this patch automatically.

In [20]:
from ctgan.synthesizers import ctgan as _ctgan_module

_OriginalDiscriminator = _ctgan_module.Discriminator

def _spectral_norm_init(self, input_dim, discriminator_dim, pac=10):
    nn.Module.__init__(self)
    dim = input_dim * pac
    self.pac = pac
    self.pacdim = dim
    seq = []
    for item in list(discriminator_dim):
        seq += [nn.utils.spectral_norm(nn.Linear(dim, item)), nn.LeakyReLU(0.2), nn.Dropout(0.5)]
        dim = item
    seq += [nn.utils.spectral_norm(nn.Linear(dim, 1))]
    self.seq = nn.Sequential(*seq)


SpectralNormDiscriminator = type(
    "SpectralNormDiscriminator", (_OriginalDiscriminator,), {"__init__": _spectral_norm_init}
)

_ctgan_module.Discriminator = SpectralNormDiscriminator


# ── Config-driven model factory ─────────────────────────────────────────────
# Each build_* function now takes an explicit `config` dict of synthesizer kwargs, rather than
# hardcoding them, so the HPO search in Section 5.5 can construct arbitrarily-configured
# synthesizers via the same factory. The DEFAULT_*_CONFIG dicts below capture the values that
# were previously hardcoded here — kept as a reproducible reference point. The Section 6
# training loop no longer uses these directly: it trains with the HPO-selected best config
# per model instead (see Section 5.5 / 6).

def build_ctgan(metadata: SingleTableMetadata, config: dict) -> CTGANSynthesizer:
    return CTGANSynthesizer(metadata, **config)


def build_tvae(metadata: SingleTableMetadata, config: dict) -> TVAESynthesizer:
    return TVAESynthesizer(metadata, **config)


def build_copulagan(metadata: SingleTableMetadata, config: dict) -> CopulaGANSynthesizer:
    return CopulaGANSynthesizer(metadata, **config)


FACTORY = {"ctgan": build_ctgan, "tvae": build_tvae, "copulagan": build_copulagan}

DEFAULT_CTGAN_CONFIG = dict(
    epochs=300,
    generator_lr=2e-4,
    generator_decay=1e-6,
    discriminator_lr=8e-4,       # TTUR: 4x generator lr
    discriminator_decay=1e-6,
    discriminator_steps=5,       # WGAN convention: more critic steps
    pac=1,                       # pac=1 -> per-sample gradient penalty
    batch_size=500,
    verbose=True,
)

DEFAULT_TVAE_CONFIG = dict(
    epochs=300,
    batch_size=500,
    l2scale=1e-5,
)

DEFAULT_COPULAGAN_CONFIG = dict(
    epochs=300,
    batch_size=500,
    generator_lr=2e-4,
    generator_decay=1e-6,
    discriminator_lr=8e-4,       # TTUR, consistent with build_ctgan
    discriminator_decay=1e-6,
)

DEFAULT_CONFIGS = {
    "ctgan": DEFAULT_CTGAN_CONFIG,
    "tvae": DEFAULT_TVAE_CONFIG,
    "copulagan": DEFAULT_COPULAGAN_CONFIG,
}

print("Model factory ready (config-driven).")

## 5.5. Hyperparameter Search

Replaces the previous fixed-hyperparameter training with a small random-search HPO loop per
model (CTGAN / TVAE / CopulaGAN). For each model:

1. Sample `N_TRIALS` configs from a conservative search space (reproducible via `HPO_SEED`).
2. For each config, fit at **reduced volume** (`N_EXPERIMENTS_TRIAL` experiments instead of the
   full `N_EXPERIMENTS`) across `SEEDS`, to get a noise estimate per config.
3. Score each trial on: TSTR R²/RMSE on a held-out **validation** split, an SDMetrics quality score, the biologically-impossible-sample rate on the raw
   (pre-validation) output, and the validation-pipeline retention rate.
4. Rank configs by a composite, rank-normalized score and retrain the winning config at full
   volume in Section 6.

In [21]:
import ast
import inspect

from sdmetrics.reports.single_table import QualityReport

# ── Search spaces ────────────────────────────────────────────────────────────────────
SEARCH_SPACES = {
    "ctgan": {
        "batch_size": [16, 32, 64],
        "generator_lr": ("loguniform", 5e-5, 5e-4),
        "discriminator_lr": ("loguniform", 5e-5, 5e-4),
        "generator_dim": [(64, 64), (128, 64), (128, 128), (256, 128)],
        "discriminator_dim": [(64, 64), (128, 64), (128, 128), (256, 128)],
        "embedding_dim": [16, 32, 64],
        "epochs": [100, 150, 200],
        "generator_decay": [0, 1e-6, 1e-5, 1e-4],
        "discriminator_decay": [0, 1e-6, 1e-5, 1e-4],
        "pac": [1],
        "discriminator_steps": [5],
    },
    "tvae": {
        "batch_size": [16, 32, 64],
        "compress_dims": [(64, 64), (128, 64), (128, 128)],
        "decompress_dims": [(64, 64), (128, 64), (128, 128)],
        "embedding_dim": [16, 32, 64],
        "epochs": [100, 150, 200],
        "l2scale": [0, 1e-6, 1e-5, 1e-4],
    },
}
# CopulaGAN wraps CTGAN internally and accepts the same constructor kwargs as CTGANSynthesizer
SEARCH_SPACES["copulagan"] = dict(SEARCH_SPACES["ctgan"])

# Verify every kwarg name in the search spaces is actually accepted by the corresponding
# SDV synthesizer constructor (sdv==1.37.2), so a typo'd kwarg doesn't silently get swallowed
# or raise deep inside a trial.
_SYNTH_CLASSES = {"ctgan": CTGANSynthesizer, "tvae": TVAESynthesizer, "copulagan": CopulaGANSynthesizer}
for _model_name, _space in SEARCH_SPACES.items():
    _accepted = set(inspect.signature(_SYNTH_CLASSES[_model_name].__init__).parameters.keys())
    _unknown = set(_space.keys()) - _accepted
    assert not _unknown, f"{_model_name}: unknown kwargs in search space: {_unknown}"
print("Search space kwargs verified against installed sdv==1.37.2 constructors.")


# ── Config sampler ───────────────────────────────────────────────────────────────────
HPO_SEED = 123  # distinct from the modeling random_state=42 used elsewhere in the notebook
N_TRIALS = 6    
N_SEEDS_PER_TRIAL = 1
SEEDS = [0]


def sample_config(space: dict, rng: np.random.Generator) -> dict:
    cfg = {}
    for k, v in space.items():
        if isinstance(v, tuple) and v[0] == "loguniform":
            lo, hi = v[1], v[2]
            cfg[k] = float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
        else:
            idx = rng.integers(len(v))
            cfg[k] = v[idx]
    return cfg


def sample_trials(model_name: str, n_trials: int, seed: int) -> list[dict]:
    rng = np.random.default_rng(seed)
    return [sample_config(SEARCH_SPACES[model_name], rng) for _ in range(n_trials)]


# Reproducibility check: regenerating the trial list twice with the same HPO_SEED must be
# identical (an actual assertion, not just a claim).
for _model_name in SEARCH_SPACES:
    _trials_a = sample_trials(_model_name, N_TRIALS, HPO_SEED)
    _trials_b = sample_trials(_model_name, N_TRIALS, HPO_SEED)
    assert _trials_a == _trials_b, f"HPO sampler is not reproducible for {_model_name}!"
print(f"HPO config sampler reproducibility verified for all models (HPO_SEED={HPO_SEED}).")

TRIALS_BY_MODEL = {name: sample_trials(name, N_TRIALS, HPO_SEED) for name in SEARCH_SPACES}
for name, trials in TRIALS_BY_MODEL.items():
    print(f"{name}: sampled {len(trials)} trial configs")


# ── Train / val / test split ─────────────────────────────────────────────────────────
X_real, y_real = prepare_split(df_raw)

X_train_real, X_test, y_train_real, y_test = train_test_split(
    X_real, y_real, test_size=0.2, random_state=42,
)
# Carve 20% of the total (25% of the 80% train partition) out of X_train_real for HPO
X_train_hpo, X_val_hpo, y_train_hpo, y_val_hpo = train_test_split(
    X_train_real, y_train_real, test_size=0.25, random_state=42,
)

print(f"X_train_hpo: {X_train_hpo.shape}, X_val_hpo: {X_val_hpo.shape}, X_test: {X_test.shape}")

# ── Integrity checks ─────────────────────────────────────────────────────────────────
assert set(X_val_hpo.index).isdisjoint(set(X_test.index)), "val/test overlap detected!"
print("Verified: val and test sets are disjoint.")

test_index_hash = hash(tuple(sorted(X_test.index)))
print(f"X_test index hash (for cross-run comparison): {test_index_hash}")


# avg_rows_per_experiment is needed here (for N_SYNTHETIC_TRIAL) and again in Section 6
# (for the full-volume N_SYNTHETIC) -- computed once here since this is the earlier use site;
# Section 6 reuses this same variable rather than recomputing it.
avg_rows_per_experiment = len(df_raw) / df_raw["experiment_id"].nunique()
print(f"avg rows/experiment in real data: {avg_rows_per_experiment:.1f}")


# ── Reduced trial volume ─────────────────────────────────────────────────────────────
N_EXPERIMENTS_TRIAL = 50  # vs. the full N_EXPERIMENTS = 1000
N_SYNTHETIC_TRIAL = int(round(N_EXPERIMENTS_TRIAL * avg_rows_per_experiment))
print(f"N_EXPERIMENTS_TRIAL={N_EXPERIMENTS_TRIAL} -> N_SYNTHETIC_TRIAL={N_SYNTHETIC_TRIAL} rows")


# ── Biologically-impossible-sample rate ──────────────────────────────────────────────
def impossible_sample_rate(df_syn_raw: pd.DataFrame) -> float:
    """
    Fraction of raw (pre-validation-rejection) generated rows that violate a biologically
    impossible condition. Reuses rule_hybrid_without_hurdle / rule_shrimp_with_packaging
    from the Shared Utilities section (Section 4.5) — these operate on ORIGINAL-scale,
    post-processed columns, so this function expects a POST-PROCESSED (not raw-scaled) frame,
    which is what postprocess() produces before validate_synthetic() is applied.

    Also adds two growth-invariant rules, computed directly here since they are new (not
    defined in Section 7.1):
      - post_threshold_count < initial_count_(day_0): population can't fall below the
        starting inoculum by the time it crosses the upper threshold (violates monotonic
        growth).
      - post_threshold_(day) < pre_threshold_(day): the post-threshold crossing can't occur
        before the pre-threshold crossing (violates monotonic time ordering).
    """
    n = len(df_syn_raw)
    if n == 0:
        return float("nan")

    violation = pd.Series(False, index=df_syn_raw.index)

    if all(c in df_syn_raw.columns for c in ["hurdle_coordination", "physical_hurdle_tech", "application_method"]):
        violation |= rule_hybrid_without_hurdle(df_syn_raw)
    if all(c in df_syn_raw.columns for c in ["post_threshold_count", "initial_count_(day_0)"]):
        violation |= (df_syn_raw["post_threshold_count"] < df_syn_raw["initial_count_(day_0)"])
    if all(c in df_syn_raw.columns for c in ["post_threshold_(day)", "pre_threshold_(day)"]):
        violation |= (df_syn_raw["post_threshold_(day)"] < df_syn_raw["pre_threshold_(day)"])

    return float(violation.mean())


# ── SDMetrics quality score ──────────────────────────────────────────────────────────
_gan_cols_metadata = SingleTableMetadata()
_gan_cols_metadata.detect_from_dataframe(df_raw[GAN_COLS])
for _col in CAT_COLS:
    if _col in GAN_COLS:
        _gan_cols_metadata.update_column(_col, sdtype="categorical")
for _col in NUM_COLS:
    if _col in GAN_COLS:
        _gan_cols_metadata.update_column(_col, sdtype="numerical")
_gan_cols_metadata.validate()
GAN_COLS_METADATA_DICT = _gan_cols_metadata.to_dict()


def compute_sdmetrics_score(df_syn: pd.DataFrame, df_real: pd.DataFrame, metadata_dict: dict) -> float:
    try:
        report = QualityReport()
        cols = [c for c in GAN_COLS if c in df_real.columns and c in df_syn.columns]
        report.generate(df_real[cols], df_syn[cols], metadata_dict, verbose=False)
        return report.get_score()
    except Exception as exc:
        print(f"SDMetrics failed: {exc}")
        return float("nan")


print("HPO setup ready: search spaces, sampler, splits, impossible_sample_rate, compute_sdmetrics_score.")

In [22]:
# ── Timing estimate ──────────────────────────────────────────────────────────────────
# Time a single small trial (1 config, 1 seed, reduced volume, low epoch count) before
# launching the full grid, purely for visibility into compute cost. 
_timing_config = dict(SEARCH_SPACES["ctgan"])
_timing_probe_config = {
    "batch_size": 32,
    "generator_lr": 2e-4,
    "discriminator_lr": 8e-4,
    "generator_dim": (64, 64),
    "discriminator_dim": (64, 64),
    "embedding_dim": 16,
    "epochs": 50,
    "generator_decay": 1e-6,
    "discriminator_decay": 1e-6,
    "pac": 1,
    "discriminator_steps": 5,
}

_t0 = time.time()
torch.manual_seed(0)
np.random.seed(0)
_probe_synth = build_ctgan(metadata, _timing_probe_config)
_probe_synth.fit(df_gan)
_probe_synth.sample(num_rows=N_SYNTHETIC_TRIAL)
_probe_elapsed = time.time() - _t0

_epochs_ratio = float(np.mean(SEARCH_SPACES["ctgan"]["epochs"])) / _timing_probe_config["epochs"]
_estimated_per_trial = _probe_elapsed * _epochs_ratio
_estimated_total_seconds = _estimated_per_trial * N_TRIALS * N_SEEDS_PER_TRIAL * len(SEARCH_SPACES)

print(f"Probe trial (batch_size=32, {_timing_probe_config['epochs']} epochs): {_probe_elapsed:.1f}s")
print(f"Estimated avg time per trial (scaled to mean epochs in search space): {_estimated_per_trial:.1f}s")
print(
    f"Estimated TOTAL wall-clock for full HPO grid "
    f"({len(SEARCH_SPACES)} models x {N_TRIALS} trials x {N_SEEDS_PER_TRIAL} seeds "
    f"= {len(SEARCH_SPACES) * N_TRIALS * N_SEEDS_PER_TRIAL} fits): "
    f"~{_estimated_total_seconds/60:.0f} min (~{_estimated_total_seconds/3600:.1f} h)"
)
print("Proceeding with the full search regardless of this estimate (visibility only, no auto-abort).")

In [ ]:
# ── Trial runner ──────────────────────────────────────────────────────────────────────
def run_trial(model_name: str, config: dict, seed: int) -> dict:
    """
    Fit one (model, config, seed) combination at reduced volume, run it through the full
    postprocess -> validate -> evaluate pipeline, and return a flat metrics dict. Any
    exception is caught so a single bad config can't crash the whole search; failures are
    logged and recorded as NaN metrics.
    """
    result = {
        "model": model_name, "seed": seed,
        "tstr_r2": np.nan, "tstr_rmse": np.nan,
        "sdmetrics_score": np.nan, "impossible_rate": np.nan, "valid_rate": np.nan,
        "n_generated": np.nan, "n_retained": np.nan,
        "status": "ok", "error": "",
    }
    try:
        torch.manual_seed(seed)
        np.random.seed(seed)

        synth = FACTORY[model_name](metadata, config)
        synth.fit(df_gan)
        df_syn_raw = synth.sample(num_rows=N_SYNTHETIC_TRIAL)

        df_syn_post = postprocess(df_syn_raw)
        result["impossible_rate"] = impossible_sample_rate(df_syn_post)

        df_valid, report = validate_synthetic(df_syn_post, df_raw)
        result["valid_rate"] = report["n_retained"] / report["n_generated"] if report["n_generated"] else np.nan
        result["n_generated"] = report["n_generated"]
        result["n_retained"] = report["n_retained"]

        X_syn, y_syn = prepare_split(df_valid)
        if len(X_syn) < 10:
            raise ValueError(f"too few retained rows with target value ({len(X_syn)}) to train TSTR model")

        pipe = make_pipeline()
        pipe.fit(X_syn, y_syn)
        tstr_metrics = evaluate(pipe, X_val_hpo, y_val_hpo)
        result["tstr_r2"] = tstr_metrics["R2"]
        result["tstr_rmse"] = tstr_metrics["RMSE"]

        result["sdmetrics_score"] = compute_sdmetrics_score(df_valid, df_raw, GAN_COLS_METADATA_DICT)

    except Exception as exc:
        result["status"] = "failed"
        result["error"] = str(exc)
        print(f"  [WARNING] trial failed (model={model_name}, seed={seed}): {exc}")

    return result


# ── Main HPO loop ─────────────────────────────────────────────────────────────────────
hpo_trial_results: dict[str, pd.DataFrame] = {}
_hpo_t0 = time.time()

for model_name, trial_configs in TRIALS_BY_MODEL.items():
    print(f"\n{'='*70}\n  HPO search: {model_name.upper()} ({len(trial_configs)} trials x {len(SEEDS)} seeds)\n{'='*70}")
    rows = []
    for trial_idx, config in enumerate(trial_configs):
        for seed in SEEDS:
            print(f"  trial {trial_idx+1}/{len(trial_configs)}, seed={seed}: {config}")
            t0 = time.time()
            row = run_trial(model_name, config, seed)
            row["trial_idx"] = trial_idx
            row["config"] = config
            row["elapsed_sec"] = time.time() - t0
            rows.append(row)
            print(f"    -> status={row['status']} tstr_r2={row['tstr_r2']:.3f} "
                  f"sdmetrics={row['sdmetrics_score']:.3f} valid_rate={row['valid_rate']:.3f} "
                  f"impossible_rate={row['impossible_rate']:.3f} ({row['elapsed_sec']:.0f}s)"
                  if row["status"] == "ok" else f"    -> status={row['status']} ({row['elapsed_sec']:.0f}s)")

    df_trials = pd.DataFrame(rows)
    hpo_trial_results[model_name] = df_trials

    trials_path = SYNTHETIC_DIR / f"hpo_trials_{model_name}.csv"
    df_trials.to_csv(trials_path, index=False)
    print(f"Saved {len(df_trials)} trial x seed rows to {trials_path}")

_hpo_elapsed = time.time() - _hpo_t0
print(f"\nTotal HPO search wall-clock so far: {_hpo_elapsed/60:.1f} min ({_hpo_elapsed/3600:.2f} h)")

In [ ]:
# ── Composite ranking ─────────────────────────────────────────────────────────────────
# Aggregate per-seed metrics to per-config means/stds, then rank via percentile-rank
# normalization per metric (robust to outlier trials, unlike raw min-max scaling).
COMPOSITE_WEIGHTS = {"r2": 0.4, "rmse": 0.2, "quality": 0.2, "validity": 0.15, "stability": 0.05}
TIEBREAK_EPSILON = 0.01


def _arch_size(config: dict) -> int:
    """Sum of hidden-dim tuple products + embedding_dim -- used as a tiebreak (smaller preferred)."""
    total = int(config.get("embedding_dim", 0))
    for key in ("generator_dim", "discriminator_dim", "compress_dims", "decompress_dims"):
        val = config.get(key)
        if val is not None:
            total += int(np.prod(val))
    return total


def rank_trials(df_trials: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-seed rows to per-config summary rows and compute a composite score."""
    ok = df_trials[df_trials["status"] == "ok"].copy()
    if ok.empty:
        raise ValueError("No successful trials to rank -- all trials failed.")

    agg = ok.groupby("trial_idx").agg(
        tstr_r2_mean=("tstr_r2", "mean"),
        tstr_r2_std=("tstr_r2", "std"),
        tstr_rmse_mean=("tstr_rmse", "mean"),
        sdmetrics_score_mean=("sdmetrics_score", "mean"),
        valid_rate_mean=("valid_rate", "mean"),
        impossible_rate_mean=("impossible_rate", "mean"),
        n_seeds_ok=("seed", "count"),
    ).reset_index()
    agg["tstr_r2_std"] = agg["tstr_r2_std"].fillna(0.0)

    # config is identical across seeds for the same trial_idx -- take the first
    config_lookup = ok.drop_duplicates("trial_idx").set_index("trial_idx")["config"]
    agg["config"] = agg["trial_idx"].map(config_lookup)
    agg["arch_size"] = agg["config"].apply(_arch_size)

    n = len(agg)
    def pct_rank(s: pd.Series) -> pd.Series:
        return s.rank(method="average", pct=True) if n > 1 else pd.Series(1.0, index=s.index)

    rank_r2 = pct_rank(agg["tstr_r2_mean"])                       # higher better
    rank_rmse = 1.0 - pct_rank(agg["tstr_rmse_mean"])              # lower better
    rank_quality = pct_rank(agg["sdmetrics_score_mean"])           # higher better
    validity_raw = agg["valid_rate_mean"] * (1.0 - agg["impossible_rate_mean"].fillna(0.0))
    rank_validity = pct_rank(validity_raw)                         # higher better
    stability_raw = 1.0 / (1.0 + agg["tstr_r2_std"])
    rank_stability = pct_rank(stability_raw)                       # higher better

    agg["composite"] = (
        COMPOSITE_WEIGHTS["r2"] * rank_r2
        + COMPOSITE_WEIGHTS["rmse"] * rank_rmse
        + COMPOSITE_WEIGHTS["quality"] * rank_quality
        + COMPOSITE_WEIGHTS["validity"] * rank_validity
        + COMPOSITE_WEIGHTS["stability"] * rank_stability
    )

    # Sort descending by composite; on near-ties (within TIEBREAK_EPSILON), prefer smaller
    # architecture size.
    agg = agg.sort_values("trial_idx").reset_index(drop=True)
    best_composite = agg["composite"].max()
    agg["near_best"] = agg["composite"] >= (best_composite - TIEBREAK_EPSILON)
    agg = agg.sort_values(
        by=["near_best", "composite", "arch_size"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    # Refine ordering: among near-best rows, smaller arch_size wins; ties broken, then order
    # remaining rows purely by composite descending.
    near_best_rows = agg[agg["near_best"]].sort_values("arch_size", ascending=True)
    rest_rows = agg[~agg["near_best"]].sort_values("composite", ascending=False)
    ranking_df = pd.concat([near_best_rows, rest_rows], ignore_index=True).drop(columns=["near_best"])

    return ranking_df


hpo_rankings: dict[str, pd.DataFrame] = {}

for model_name, df_trials in hpo_trial_results.items():
    ranking_df = rank_trials(df_trials)
    hpo_rankings[model_name] = ranking_df

    ranking_path = SYNTHETIC_DIR / f"hpo_ranking_{model_name}.csv"
    ranking_df.to_csv(ranking_path, index=False)
    print(f"\n{model_name.upper()} ranking (top 3):")
    print(ranking_df[["trial_idx", "composite", "tstr_r2_mean", "tstr_rmse_mean",
                        "sdmetrics_score_mean", "valid_rate_mean", "arch_size"]].head(3).to_string(index=False))
    print(f"Saved ranking table to {ranking_path}")

    # ── Convergence plot ─────────────────────────────────────────────────────────────
    by_trial_order = ranking_df.sort_values("trial_idx")
    running_best = by_trial_order["composite"].cummax()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(by_trial_order["trial_idx"], by_trial_order["composite"], "o-", label="trial composite score", alpha=0.6)
    ax.plot(by_trial_order["trial_idx"], running_best.values, "s--", label="running best", color="black")
    ax.set_xlabel("Trial index")
    ax.set_ylabel("Composite score")
    ax.set_title(f"HPO convergence: {model_name.upper()}")
    ax.legend()
    plt.tight_layout()
    convergence_path = SYNTHETIC_DIR / f"hpo_convergence_{model_name}.png"
    plt.savefig(convergence_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved convergence plot to {convergence_path}")

    # Honest report on convergence: is the search still improving or has it plateaued?
    n_trials_total = len(by_trial_order)
    half = max(1, n_trials_total // 2)
    first_half_best = by_trial_order["composite"].iloc[:half].max()
    last_window = min(5, n_trials_total)
    best_in_last_window = by_trial_order["composite"].iloc[-last_window:].max()
    best_in_first_window = by_trial_order["composite"].iloc[:last_window].max()
    still_improving = best_in_last_window > best_in_first_window + 1e-9
    print(
        f"{model_name.upper()}: best composite in first {last_window} trials = {best_in_first_window:.4f}, "
        f"best in last {last_window} trials = {best_in_last_window:.4f} -> "
        f"{'still improving' if still_improving else 'plateaued'} by trial {n_trials_total} "
        f"(with only {n_trials_total} trials sampled, this is a weak signal either way)."
    )

## 6. Training Loop (HPO-tuned)

Each model is trained with its Section-5.5 HPO-selected best config. 
The GAN trains on `df_gan` (source columns only, no IDs, no derived columns). After fitting 
it generates enough rows to produce the full `N_EXPERIMENTS`
synthetic experiment groups (estimated from the real data's average rows-per-experiment).
Raw GAN output is saved to `data/synthetic/hpo_original/<model_name>_raw.csv` before
post-processing.

In [ ]:
# ── Section 6: Retrain with best HPO config, at full volume ──────────────────────────
# For each model, take the top composite-score row from its ranking table and retrain at
# full volume (N_EXPERIMENTS / N_SYNTHETIC, the original 1000-experiment target).
HPO_ORIGINAL_DIR = SYNTHETIC_DIR / "hpo_original"
HPO_ORIGINAL_DIR.mkdir(parents=True, exist_ok=True)

BEST_CONFIGS: dict[str, dict] = {}
for model_name, ranking_df in hpo_rankings.items():
    best_row = ranking_df.iloc[0]
    best_config = dict(best_row["config"])
    BEST_CONFIGS[model_name] = best_config
    print(f"{model_name.upper()} best config (trial {int(best_row['trial_idx'])}, "
          f"composite={best_row['composite']:.4f}): {best_config}")

for model_name in BEST_CONFIGS:
    _reloaded = pd.read_csv(SYNTHETIC_DIR / f"hpo_ranking_{model_name}.csv")
    _config_from_csv = ast.literal_eval(_reloaded.iloc[0]["config"])
    assert isinstance(_config_from_csv, dict), f"{model_name}: config did not round-trip via ast.literal_eval"
print("Verified: best configs round-trip correctly through the saved CSV (ast.literal_eval).")

N_SYNTHETIC = int(round(N_EXPERIMENTS * avg_rows_per_experiment))
print(f"Target synthetic rows (full volume): {N_SYNTHETIC} (= {N_EXPERIMENTS} experiments x {avg_rows_per_experiment:.1f})")

EXPERIMENTS = {
    name: FACTORY[name](metadata, BEST_CONFIGS[name])
    for name in BEST_CONFIGS
}

synthetic_raw: dict[str, pd.DataFrame] = {}

for name, synth in tqdm(EXPERIMENTS.items(), total=len(EXPERIMENTS), desc="Retraining with best HPO configs"):
    print(f"\n{'='*60}")
    print(f"  Training {name.upper()} (best HPO config, {BEST_CONFIGS[name].get('epochs')} epochs)")
    print(f"{'='*60}")
    t0 = time.time()
    torch.manual_seed(42)
    np.random.seed(42)
    synth.fit(df_gan)
    print(f"  -> Fit complete in {(time.time() - t0) / 60:.1f} min")

    df_syn_raw = synth.sample(num_rows=N_SYNTHETIC)
    raw_path = HPO_ORIGINAL_DIR / f"{name}_raw.csv"
    df_syn_raw.to_csv(raw_path, index=False)
    synthetic_raw[name] = df_syn_raw
    print(f"  -> Saved {N_SYNTHETIC} raw rows to {raw_path}")

print("\nAll HPO-tuned full-volume retrains complete.")

## 7. Post-processing: Re-assign IDs & Recompute Derived Columns

Four steps applied to every raw GAN output before saving the final file:

1. **Inverse-transform numerical features** — GAN output for `NUM_COLS` is on the
   `MinMaxScaler`-fitted [0, 1] scale (Section 3); apply `NUM_SCALER.inverse_transform`
   to bring every numerical column back to its original units before anything downstream
   (derived-column formulas, validation, clipping) touches it.
2. **Drop indicator columns** — `*_missing` / `*_censored` were training-only artifacts
   that let CTGAN learn the real data's missingness pattern.
3. **Re-assign `experiment_id`** — rows sharing the same `EXPERIMENT_KEY` values
   (meat_type, indicators, storage condition, treatment) receive the same sequential
   integer ID, starting from the next integer after the real data's maximum.
   **Re-assign `ingredient_id`** — within each synthetic experiment, each unique
   `ingredient` gets a distinct sequential integer (1, 2, 3 …).
4. **Recompute derived columns** — `initial_inhibition_factor`, `post_inhibition_factor`,
   and `post_threshold_proximity_index_2(tpi)` are computed from the GAN-generated
   (now inverse-transformed) source columns using the same formulas as
   `data_processing.ipynb`. `normalized_concentration_%_of_meat` has no available formula
   and is left `NaN`. `data_source` is set to `"synthetic"` for every row.

Outputs from this HPO-tuned run are saved to `data/synthetic/hpo_original/`.

In [ ]:
synthetic_datasets: dict[str, pd.DataFrame] = {}

for name, df_syn_raw in synthetic_raw.items():
    df_syn = postprocess(df_syn_raw)
    out_path = HPO_ORIGINAL_DIR / f"{name}.csv"
    df_syn.to_csv(out_path, index=False)
    synthetic_datasets[name] = df_syn
    print(f"{name}: {df_syn.shape[0]} rows saved to {out_path}")
    print(f"  experiment_id range : {df_syn['experiment_id'].min()} - {df_syn['experiment_id'].max()}")
    print(f"  unique experiment_id: {df_syn['experiment_id'].nunique()}")
    print(f"  data_source values  : {df_syn['data_source'].unique()}")
    print(f"  sample derived values:")
    sample = df_syn[["initial_inhibition_factor", "post_inhibition_factor",
                     "post_threshold_proximity_index_2(tpi)"]].head(3)
    print(sample.to_string(index=False))

## 7.1. Post-Generation Validation

1. **Logical consistency** — each rule is a small predicate function registered in a list,
   so new rules can be added without touching the orchestrator. Two rules are hard-rejected
   because they are directly evidenced by the real data:
   - `hurdle_coordination == "Hybrid_Simultaneous"` with both `physical_hurdle_tech` and
     `application_method` empty (the coordination axis claims multiple hurdles were applied,
     but neither hurdle column says what they were).
   A third check — co-occurrence novelty — is **reported, not rejected**, by default: the set
   of `(packaging_class, packaging_material, container_type)` and
   `(physical_hurdle_tech, application_method, hurdle_coordination)` tuples seen in the real
   data is used as a discovered whitelist, and synthetic rows with an unseen tuple are counted
   but kept (CTGAN's value partly comes from interpolating between observed combinations, so
   hard-rejecting every novel tuple would defeat that). Set `STRICT_NOVELTY = True` (Section 4.5)
   to reject novel tuples too.
2. **Clipping** — every numerical column in `NUM_COLS` is clipped to the real data's
   1st–99th percentile range.
3. **Reporting** — prints total generated, retained, rejected, and a per-rule rejection
   breakdown, then re-runs the two hard rules on the retained set and asserts zero violations
   remain.

In [ ]:
validated_datasets: dict[str, pd.DataFrame] = {}

for name, df_syn in synthetic_datasets.items():
    print(f"\n{'='*60}\n  Validating {name.upper()}\n{'='*60}")
    df_valid, report = validate_synthetic(df_syn, df_raw)
    validation_report(report)
    validated_datasets[name] = df_valid

    # Sanity check: zero hard-rule violations remain in the retained set
    remaining_violations = {
        rule_name: int(rule_fn(df_valid).sum())
        for rule_name, rule_fn in HARD_REJECT_RULES.items()
    }
    assert all(v == 0 for v in remaining_violations.values()), (
        f"Hard-rule violations survived validation: {remaining_violations}"
    )
    print("\nSanity check passed: zero hard-rule violations in retained set.")

### 7.2. Final Output

Concatenate the validated, clipped, gas-reconstructed output of every model into a single
file. This is additive — the per-model raw (`<name>_raw.csv`) and post-processed
(`<name>.csv`) files from Section 7 are kept as-is for debugging; `validated_datasets`
(used by this cell) is what feeds the final deliverable.

In [ ]:
synthetic_data_final = pd.concat(validated_datasets.values(), ignore_index=True)

final_path = HPO_ORIGINAL_DIR / "synthetic_data_final.csv"
synthetic_data_final.to_csv(final_path, index=False)
print(f"Saved {len(synthetic_data_final):,} validated rows to {final_path}")

## 8. TSTR Evaluation

**Train on Synthetic, Test on Real** protocol.

Target: `post_threshold_proximity_index_2(tpi)` — a continuous score capturing how far a
preservation outcome has crossed the upper safety threshold.

- **Baseline** — trained on real data, tested on the fixed 20% real hold-out (`X_test`/`y_test`,
  computed once in Section 5.5 and reused here without modification).
- **CTGAN / TVAE / CopulaGAN** — trained on the corresponding HPO-tuned, full-volume synthetic
  dataset (Section 6), tested on the same hold-out.

Metrics: RMSE, MAE, R².

In [ ]:
results: dict[str, dict] = {}

pipe_real = make_pipeline()
pipe_real.fit(X_train_real, y_train_real)
results["real (baseline)"] = evaluate(pipe_real, X_test, y_test)
print("Baseline (real -> real):", results["real (baseline)"])

for name, df_syn in synthetic_datasets.items():
    X_syn, y_syn = prepare_split(df_syn)
    if len(X_syn) == 0:
        print(f"{name.upper()}: no rows with target value after post-processing -- skipping")
        continue
    pipe_syn = make_pipeline()
    pipe_syn.fit(X_syn, y_syn)
    results[name] = evaluate(pipe_syn, X_test, y_test)
    print(f"{name.upper()} (syn -> real):", results[name])

## 9. Results Summary

In [ ]:
baseline_rmse = results["real (baseline)"]["RMSE"]

df_results = (
    pd.DataFrame(results).T
    .rename_axis("experiment")
    .reset_index()
)
df_results["RMSE_vs_baseline_%"] = (
    (df_results["RMSE"] - baseline_rmse) / baseline_rmse * 100
).round(1)
df_results = df_results.sort_values("RMSE").round(4)

print("\n=== TSTR Results (HPO-tuned, sorted by RMSE) ===")
print(df_results.to_string(index=False))

df_results.to_csv(HPO_ORIGINAL_DIR / "tstr_results.csv", index=False)
print(f"\nSaved to {HPO_ORIGINAL_DIR / 'tstr_results.csv'}")

In [ ]:
metrics = ["RMSE", "MAE", "R2"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0"]

for ax, metric in zip(axes, metrics):
    bars = ax.bar(df_results["experiment"], df_results[metric], color=colors[:len(df_results)])
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_xticks(range(len(df_results)))
    ax.set_xticklabels(df_results["experiment"], rotation=15, ha="right")
    for bar, val in zip(bars, df_results[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.01,
            f"{val:.3f}",
            ha="center", va="bottom", fontsize=9,
        )

fig.suptitle(
    "TSTR Evaluation (HPO-tuned) - post_threshold_proximity_index prediction",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.savefig(HPO_ORIGINAL_DIR / "tstr_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved.")

## 10. Statistical Fidelity Check

Column-wise KS statistics between real and synthetic numerical distributions.
Lower values indicate better marginal fidelity, independently of predictive utility.

In [ ]:
# ks_fidelity() is defined once in Section 4.5 (Shared Utilities) and reused here.
ks_rows = {name: ks_fidelity(df_raw, df_syn) for name, df_syn in synthetic_datasets.items()}
df_ks = pd.DataFrame(ks_rows)
df_ks.loc["mean"] = df_ks.mean()

print("=== KS Statistics (HPO-tuned, lower = better) ===")
print(df_ks.to_string())
df_ks.to_csv(HPO_ORIGINAL_DIR / "ks_fidelity.csv")
print(f"\nSaved to {HPO_ORIGINAL_DIR / 'ks_fidelity.csv'}")